# Task 3 – K-Nearest Neighbours (KNN) Classifier

**Goal:** Classify wine varieties from chemical measurements and find the optimal K.

**How KNN works:**  
For each new sample, the algorithm looks at the K closest training points and takes a majority vote. Small K = noisy but flexible; large K = smooth but may underfit. We test K = 1, 3, 5, 7, 9 and pick the winner.

**Dataset:** sklearn's Wine dataset (178 samples, 13 features, 3 classes)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 1. Load & Preprocess Data

KNN uses Euclidean distance, so feature scaling is essential — otherwise a feature with a large range (like Proline, which can be 400–1700) would dominate all distance calculations.

In [ ]:
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print(f"Samples: {X.shape[0]}  |  Features: {X.shape[1]}  |  Classes: {len(set(y))}")

# Scale before splitting so there's no data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

## 2. Find the Best K

In [ ]:
k_values = [1, 3, 5, 7, 9]
scores   = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test))
    scores.append(acc)
    print(f"K = {k:>2}  →  Accuracy: {acc:.4f}")

best_k = k_values[scores.index(max(scores))]
print(f"\nBest K: {best_k} with accuracy {max(scores):.4f}")

## 3. Visualise K vs Accuracy

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(k_values, scores, marker='o', linewidth=2, color='steelblue')
plt.xticks(k_values)
plt.xlabel("Number of Neighbours (K)")
plt.ylabel("Test Accuracy")
plt.title("KNN: Accuracy vs K")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Evaluate the Best Model

In [ ]:
best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_knn.fit(X_train, y_train)
final_preds = best_knn.predict(X_test)

print(f"Classification Report (K={best_k}):")
print(classification_report(final_preds, y_test, target_names=data.target_names))

## 5. Confusion Matrix

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    confusion_matrix(y_test, final_preds),
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=data.target_names,
    yticklabels=data.target_names
)
plt.title(f"Confusion Matrix – KNN (K={best_k})")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()